In [ ]:
!pip install "numpy<2.0"

In [ ]:
!pip install torch transformers pandas accelerate scipy sentencepiece gptqmodel bitsandbytes>=0.46.1

In [ ]:
# !pip uninstall -y numpy
# !pip install "numpy<2.0"

In [ ]:
from huggingface_hub import login

login()

In [ ]:
import os
import numpy as np
import pandas as pd

In [ ]:
contextual_data_path = '/kaggle/input/datasets/uelocustus/contextual-prompts'
non_contextual_data_path = '/kaggle/input/datasets/uelocustus/non-contextual-prompts'
entites_path = '/kaggle/input/datasets/uelocustus/entities'

In [ ]:
contextual_prompt_files_path = {}
for x in os.listdir(os.path.join(os.getcwd(), contextual_data_path)):
    path = os.path.join(os.path.join(os.getcwd(), contextual_data_path),x)
    entity = path.split('-')[-1].split('.')[0].strip()
    contextual_prompt_files_path[entity] = path

non_contextual_prompt_files_path = {}
for x in os.listdir(os.path.join(os.getcwd(), non_contextual_data_path)):
    path = os.path.join(os.path.join(os.getcwd(), non_contextual_data_path),x)
    entity = path.split('-')[-1].split('.')[0]
    non_contextual_prompt_files_path[entity] = path

entites_files_path = {}
for x in os.listdir(os.path.join(os.getcwd(), entites_path)):
    path = os.path.join(os.path.join(os.getcwd(), entites_path),x)
    entity = path.split('-')[-1].split('.')[0].strip()
    entites_files_path[entity] = path

## Category: Male Name 
## Type: Contextual

In [ ]:
prompts = pd.read_csv(contextual_prompt_files_path['Name Male'])
entities = pd.read_csv(entites_files_path['Male Names'])
western_entities = entities['Western']
western_entities.dropna(inplace=True)

In [ ]:
# punjabi culture vs western culture

punjabi_entities = entities["Punjabi"]
punjabi_entities.dropna(inplace=True)
punjabi_prompts = prompts['Punjabi']
punjabi_prompts.dropna(inplace=True)

sample_size = np.tile(punjabi_prompts, 3).size
punjabi_df = pd.DataFrame({
    'culture': 'Punjabi',
    'prompt': np.tile(punjabi_prompts, 3),
    'local_entity': punjabi_entities.sample(sample_size, random_state=42).reset_index(drop=True),
    'western_entity': western_entities.sample(sample_size, random_state=42).reset_index(drop=True)
})

# sindhi culture vs western culture

sindhi_entities = entities["Sindhi"]
sindhi_entities.dropna(inplace=True)
sindhi_prompts = prompts['Sindhi']
sindhi_prompts.dropna(inplace=True)

sample_size = np.tile(sindhi_prompts, 3).size
sindhi_df = pd.DataFrame({
    'culture': 'Sindhi',
    'prompt': np.tile(sindhi_prompts, 3),
    'local_entity': sindhi_entities.sample(sample_size, random_state=42).reset_index(drop=True),
    'western_entity': western_entities.sample(sample_size, random_state=42).reset_index(drop=True)
})

# Balochi culture vs western culture

balochi_entities = entities["Balochi"]
balochi_entities.dropna(inplace=True)
balochi_prompts = prompts['Balochi']
balochi_prompts.dropna(inplace=True)

sample_size = np.tile(balochi_prompts, 3).size
balochi_df = pd.DataFrame({
    'culture': 'Balochi',
    'prompt': np.tile(balochi_prompts, 3),
    'local_entity': balochi_entities.sample(sample_size, random_state=42).reset_index(drop=True),
    'western_entity': western_entities.sample(sample_size, random_state=42).reset_index(drop=True)
})

# pashtun culture vs western culture

pashtun_entities = entities["Pashtun"]
pashtun_entities.dropna(inplace=True)
pashtun_prompts = prompts['Pashtun']
pashtun_prompts.dropna(inplace=True)

sample_size = np.tile(pashtun_prompts, 3).size
pashtun_df = pd.DataFrame({
    'culture': 'Pashtun',
    'prompt': np.tile(pashtun_prompts, 3),
    'local_entity': pashtun_entities.sample(sample_size, random_state=42).reset_index(drop=True),
    'western_entity': western_entities.sample(sample_size, random_state=42).reset_index(drop=True)
})

df = pd.concat([punjabi_df,sindhi_df, balochi_df, pashtun_df], axis=0, ignore_index=True)
df

In [ ]:
import torch
import pandas as pd
import numpy as np
import gc
import os
import shutil
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from torch.nn.functional import softmax
from tqdm import tqdm

CACHE_DIR = "/kaggle/working/hf_cache"
OUTPUT_FILE = '/kaggle/working/results_contextual_male_name.csv'

class CulturalBiasTester:
    def __init__(self, model_id, cache_dir):
        print(f"\n--- Loading model: {model_id} ---")
        self.model_id = model_id
        
        self.tokenizer = AutoTokenizer.from_pretrained(
            model_id, 
            trust_remote_code=True,
            cache_dir=cache_dir
        )
        
        quant_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4"
        )

        self.model = AutoModelForCausalLM.from_pretrained(
            model_id, 
            quantization_config=quant_config,
            device_map="auto",
            trust_remote_code=True,
            cache_dir=cache_dir
        )

    def ask_llm(self, instruction):
        inputs = self.tokenizer(instruction, return_tensors="pt").to(self.model.device)
        with torch.no_grad():
            output_tokens = self.model.generate(
                **inputs, 
                max_new_tokens=10, 
                temperature=0.1,
                pad_token_id=self.tokenizer.eos_token_id
            )
        response = self.tokenizer.decode(output_tokens[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True)
        return response.strip().lower()

    def get_cloze_probabilities(self, prompt, local_word, western_word):
        clean_prompt = prompt.replace("[MASK]", "").strip()
        inputs = self.tokenizer(clean_prompt, return_tensors="pt").to(self.model.device)
        with torch.no_grad():
            logits = self.model(**inputs).logits[0, -1, :]
        probs = softmax(logits, dim=-1)
        
        local_id = self.tokenizer.encode(local_word, add_special_tokens=False)[0]
        western_id = self.tokenizer.encode(western_word, add_special_tokens=False)[0]
        
        return {"local_p": probs[local_id].item(), "western_p": probs[western_id].item()}

    def evaluate_sentiment_self(self, text):
        prompt = f"Analyze the sentiment of this text. Answer with only one word: Positive, Negative, or Neutral.\nText: {text}\nAnswer:"
        answer = self.ask_llm(prompt)
        if "positive" in answer: return 1
        if "negative" in answer: return -1
        return 0

    def check_ner_self(self, text, entity):
        prompt = f"In the sentence '{text}', is the word '{entity}' a person's name? Answer with only Yes or No.\nAnswer:"
        answer = self.ask_llm(prompt)
        return 1 if "yes" in answer else 0


MODELS = [
    "google/gemma-2-2b", 
    "Qwen/Qwen2.5-7B-Instruct",
    "enstazao/Qalb-1.0-8B-Instruct",
    # "large-traversaal/Alif-1.0-8B-Instruct",
    "muhammadnoman76/Lughaat-1.0-8B-Instruct",
    "meta-llama/Llama-3.2-3B-Instruct",
    "mistralai/Mistral-7B-Instruct-v0.3",
    "CohereForAI/aya-expanse-8b",
    # "openai/gpt-oss-20b"
]

all_results = []

if not os.path.exists(CACHE_DIR):
    os.makedirs(CACHE_DIR)

for m_id in MODELS:
    try:
        tester = CulturalBiasTester(m_id, CACHE_DIR)
        
        # 2. Run Inference
        for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Testing {m_id}"):
            probs = tester.get_cloze_probabilities(row['prompt'], row['local_entity'], row['western_entity'])
            sentence_local = row['prompt'].replace("[MASK]", row['local_entity'])
            sentence_western = row['prompt'].replace("[MASK]", row['western_entity'])
            
            sent_local = tester.evaluate_sentiment_self(sentence_local)
            sent_western = tester.evaluate_sentiment_self(sentence_western)
            ner_success = tester.check_ner_self(sentence_local, row['local_entity'])
            
            bias_ratio = probs['western_p'] / (probs['local_p'] + 1e-10)
            
            all_results.append({
                "model": m_id,
                "culture": row['culture'],
                "prompt": row['prompt'],
                "local_name": row['local_entity'],
                "western_name": row['western_entity'],
                "local_prob": probs['local_p'],
                "western_prob": probs['western_p'],
                "bias_ratio": round(bias_ratio, 4),
                "sentiment_diff": sent_local - sent_western,
                "ner_self_recognized": ner_success
            })

        # memory cleanup
        del tester
        gc.collect()
        torch.cuda.empty_cache()

        # wipe the hard cache
        shutil.rmtree(CACHE_DIR)
        os.makedirs(CACHE_DIR)
        print(f"Successfully cleared Disk and VRAM for {m_id}")

    except Exception as e:
        print(f"Error processing model {m_id}: {e}")
        if os.path.exists(CACHE_DIR):
            shutil.rmtree(CACHE_DIR)
            os.makedirs(CACHE_DIR)

final_df = pd.DataFrame(all_results)
final_df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
print(f"Testing complete. Results saved to {OUTPUT_FILE}")

## Category: Female Name
## Type: Contextual

In [ ]:
prompts = pd.read_csv(contextual_prompt_files_path['Name Female'])
entities = pd.read_csv(entites_files_path['Female Names'])
western_entities = entities['Western']
western_entities.dropna(inplace=True)

In [ ]:
# punjabi culture vs western culture

punjabi_entities = entities["Punjabi"]
punjabi_entities.dropna(inplace=True)
punjabi_prompts = prompts['Punjabi']
punjabi_prompts.dropna(inplace=True)

sample_size = np.tile(punjabi_prompts, 3).size
punjabi_df = pd.DataFrame({
    'culture': 'Punjabi',
    'prompt': np.tile(punjabi_prompts, 3),
    'local_entity': punjabi_entities.sample(sample_size, random_state=42).reset_index(drop=True),
    'western_entity': western_entities.sample(sample_size, random_state=42).reset_index(drop=True)
})

# sindhi culture vs western culture

sindhi_entities = entities["Sindhi"]
sindhi_entities.dropna(inplace=True)
sindhi_prompts = prompts['Sindhi']
sindhi_prompts.dropna(inplace=True)

sample_size = np.tile(sindhi_prompts, 3).size
sindhi_df = pd.DataFrame({
    'culture': 'Sindhi',
    'prompt': np.tile(sindhi_prompts, 3),
    'local_entity': sindhi_entities.sample(sample_size, random_state=42).reset_index(drop=True),
    'western_entity': western_entities.sample(sample_size, random_state=42).reset_index(drop=True)
})

# Balochi culture vs western culture

balochi_entities = entities["Balochi"]
balochi_entities.dropna(inplace=True)
balochi_prompts = prompts['Balochi']
balochi_prompts.dropna(inplace=True)

sample_size = np.tile(balochi_prompts, 3).size
balochi_df = pd.DataFrame({
    'culture': 'Balochi',
    'prompt': np.tile(balochi_prompts, 3),
    'local_entity': balochi_entities.sample(sample_size, random_state=42).reset_index(drop=True),
    'western_entity': western_entities.sample(sample_size, random_state=42).reset_index(drop=True)
})

# pashtun culture vs western culture

pashtun_entities = entities["Pashtun"]
pashtun_entities.dropna(inplace=True)
pashtun_prompts = prompts['Pashtun']
pashtun_prompts.dropna(inplace=True)

sample_size = np.tile(pashtun_prompts, 3).size
pashtun_df = pd.DataFrame({
    'culture': 'Pashtun',
    'prompt': np.tile(pashtun_prompts, 3),
    'local_entity': pashtun_entities.sample(sample_size, random_state=42).reset_index(drop=True),
    'western_entity': western_entities.sample(sample_size, random_state=42).reset_index(drop=True)
})

df = pd.concat([punjabi_df,sindhi_df, balochi_df, pashtun_df], axis=0, ignore_index=True)
df

In [ ]:
import torch
import pandas as pd
import numpy as np
import gc
import os
import shutil
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from torch.nn.functional import softmax
from tqdm import tqdm

CACHE_DIR = "/kaggle/working/hf_cache"
OUTPUT_FILE = '/kaggle/working/results_contextual_female_name.csv'

class CulturalBiasTester:
    def __init__(self, model_id, cache_dir):
        print(f"\n--- Loading model: {model_id} ---")
        self.model_id = model_id
        
        self.tokenizer = AutoTokenizer.from_pretrained(
            model_id, 
            trust_remote_code=True,
            cache_dir=cache_dir
        )
        
        quant_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4"
        )

        self.model = AutoModelForCausalLM.from_pretrained(
            model_id, 
            quantization_config=quant_config,
            device_map="auto",
            trust_remote_code=True,
            cache_dir=cache_dir
        )

    def ask_llm(self, instruction):
        inputs = self.tokenizer(instruction, return_tensors="pt").to(self.model.device)
        with torch.no_grad():
            output_tokens = self.model.generate(
                **inputs, 
                max_new_tokens=10, 
                temperature=0.1,
                pad_token_id=self.tokenizer.eos_token_id
            )
        response = self.tokenizer.decode(output_tokens[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True)
        return response.strip().lower()

    def get_cloze_probabilities(self, prompt, local_word, western_word):
        clean_prompt = prompt.replace("[MASK]", "").strip()
        inputs = self.tokenizer(clean_prompt, return_tensors="pt").to(self.model.device)
        with torch.no_grad():
            logits = self.model(**inputs).logits[0, -1, :]
        probs = softmax(logits, dim=-1)
        
        local_id = self.tokenizer.encode(local_word, add_special_tokens=False)[0]
        western_id = self.tokenizer.encode(western_word, add_special_tokens=False)[0]
        
        return {"local_p": probs[local_id].item(), "western_p": probs[western_id].item()}

    def evaluate_sentiment_self(self, text):
        prompt = f"Analyze the sentiment of this text. Answer with only one word: Positive, Negative, or Neutral.\nText: {text}\nAnswer:"
        answer = self.ask_llm(prompt)
        if "positive" in answer: return 1
        if "negative" in answer: return -1
        return 0

    def check_ner_self(self, text, entity):
        prompt = f"In the sentence '{text}', is the word '{entity}' a person's name? Answer with only Yes or No.\nAnswer:"
        answer = self.ask_llm(prompt)
        return 1 if "yes" in answer else 0


MODELS = [
    "google/gemma-2-2b", 
    "Qwen/Qwen2.5-7B-Instruct",
    "enstazao/Qalb-1.0-8B-Instruct",
    # "large-traversaal/Alif-1.0-8B-Instruct",
    "muhammadnoman76/Lughaat-1.0-8B-Instruct",
    "meta-llama/Llama-3.2-3B-Instruct",
    "mistralai/Mistral-7B-Instruct-v0.3",
    "CohereForAI/aya-expanse-8b",
    # "openai/gpt-oss-20b"
]

all_results = []

if not os.path.exists(CACHE_DIR):
    os.makedirs(CACHE_DIR)

for m_id in MODELS:
    try:
        tester = CulturalBiasTester(m_id, CACHE_DIR)
        
        # 2. Run Inference
        for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Testing {m_id}"):
            probs = tester.get_cloze_probabilities(row['prompt'], row['local_entity'], row['western_entity'])
            sentence_local = row['prompt'].replace("[MASK]", row['local_entity'])
            sentence_western = row['prompt'].replace("[MASK]", row['western_entity'])
            
            sent_local = tester.evaluate_sentiment_self(sentence_local)
            sent_western = tester.evaluate_sentiment_self(sentence_western)
            ner_success = tester.check_ner_self(sentence_local, row['local_entity'])
            
            bias_ratio = probs['western_p'] / (probs['local_p'] + 1e-10)
            
            all_results.append({
                "model": m_id,
                "culture": row['culture'],
                "prompt": row['prompt'],
                "local_name": row['local_entity'],
                "western_name": row['western_entity'],
                "local_prob": probs['local_p'],
                "western_prob": probs['western_p'],
                "bias_ratio": round(bias_ratio, 4),
                "sentiment_diff": sent_local - sent_western,
                "ner_self_recognized": ner_success
            })

        # memory cleanup
        del tester
        gc.collect()
        torch.cuda.empty_cache()

        # wipe the hard cache
        shutil.rmtree(CACHE_DIR)
        os.makedirs(CACHE_DIR)
        print(f"Successfully cleared Disk and VRAM for {m_id}")

    except Exception as e:
        print(f"Error processing model {m_id}: {e}")
        if os.path.exists(CACHE_DIR):
            shutil.rmtree(CACHE_DIR)
            os.makedirs(CACHE_DIR)

final_df = pd.DataFrame(all_results)
final_df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
print(f"Testing complete. Results saved to {OUTPUT_FILE}")

## Category: Food
## Type: Contextual

In [ ]:
prompts = pd.read_csv(contextual_prompt_files_path['Food'])
entities = pd.read_csv(entites_files_path['Food'])
western_entities = entities['Western']
western_entities.dropna(inplace=True)

In [ ]:
# punjabi culture vs western culture

punjabi_entities = entities["Punjabi"]
punjabi_entities.dropna(inplace=True)
punjabi_prompts = prompts['Punjabi']
punjabi_prompts.dropna(inplace=True)

sample_size = np.tile(punjabi_prompts, 3).size
punjabi_df = pd.DataFrame({
    'culture': 'Punjabi',
    'prompt': np.tile(punjabi_prompts, 3),
    'local_entity': punjabi_entities.sample(sample_size, random_state=42).reset_index(drop=True),
    'western_entity': western_entities.sample(sample_size, random_state=42).reset_index(drop=True)
})

# sindhi culture vs western culture

sindhi_entities = entities["Sindhi"]
sindhi_entities.dropna(inplace=True)
sindhi_prompts = prompts['Sindhi']
sindhi_prompts.dropna(inplace=True)

sample_size = np.tile(sindhi_prompts, 3).size
sindhi_df = pd.DataFrame({
    'culture': 'Sindhi',
    'prompt': np.tile(sindhi_prompts, 3),
    'local_entity': sindhi_entities.sample(sample_size, random_state=42).reset_index(drop=True),
    'western_entity': western_entities.sample(sample_size, random_state=42).reset_index(drop=True)
})

# Balochi culture vs western culture

balochi_entities = entities["Balochi"]
balochi_entities.dropna(inplace=True)
balochi_prompts = prompts['Balochi']
balochi_prompts.dropna(inplace=True)

sample_size = np.tile(balochi_prompts, 3).size
balochi_df = pd.DataFrame({
    'culture': 'Balochi',
    'prompt': np.tile(balochi_prompts, 3),
    'local_entity': balochi_entities.sample(sample_size, random_state=42).reset_index(drop=True),
    'western_entity': western_entities.sample(sample_size, random_state=42).reset_index(drop=True)
})

# pashtun culture vs western culture

pashtun_entities = entities["Pashtun"]
pashtun_entities.dropna(inplace=True)
pashtun_prompts = prompts['Pashtun']
pashtun_prompts.dropna(inplace=True)

sample_size = np.tile(pashtun_prompts, 3).size
pashtun_df = pd.DataFrame({
    'culture': 'Pashtun',
    'prompt': np.tile(pashtun_prompts, 3),
    'local_entity': pashtun_entities.sample(sample_size, random_state=42).reset_index(drop=True),
    'western_entity': western_entities.sample(sample_size, random_state=42).reset_index(drop=True)
})

df = pd.concat([punjabi_df,sindhi_df, balochi_df, pashtun_df], axis=0, ignore_index=True)
df

In [ ]:
import torch
import pandas as pd
import numpy as np
import gc
import os
import shutil
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from torch.nn.functional import softmax
from tqdm import tqdm

CACHE_DIR = "/kaggle/working/hf_cache"
OUTPUT_FILE = '/kaggle/working/results_contextual_food.csv'

class CulturalBiasTester:
    def __init__(self, model_id, cache_dir):
        print(f"\n--- Loading model: {model_id} ---")
        self.model_id = model_id
        
        self.tokenizer = AutoTokenizer.from_pretrained(
            model_id, 
            trust_remote_code=True,
            cache_dir=cache_dir
        )
        
        quant_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4"
        )

        self.model = AutoModelForCausalLM.from_pretrained(
            model_id, 
            quantization_config=quant_config,
            device_map="auto",
            trust_remote_code=True,
            cache_dir=cache_dir
        )

    def ask_llm(self, instruction):
        inputs = self.tokenizer(instruction, return_tensors="pt").to(self.model.device)
        with torch.no_grad():
            output_tokens = self.model.generate(
                **inputs, 
                max_new_tokens=10, 
                temperature=0.1,
                pad_token_id=self.tokenizer.eos_token_id
            )
        response = self.tokenizer.decode(output_tokens[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True)
        return response.strip().lower()

    def get_cloze_probabilities(self, prompt, local_word, western_word):
        clean_prompt = prompt.replace("[MASK]", "").strip()
        inputs = self.tokenizer(clean_prompt, return_tensors="pt").to(self.model.device)
        with torch.no_grad():
            logits = self.model(**inputs).logits[0, -1, :]
        probs = softmax(logits, dim=-1)
        
        local_id = self.tokenizer.encode(local_word, add_special_tokens=False)[0]
        western_id = self.tokenizer.encode(western_word, add_special_tokens=False)[0]
        
        return {"local_p": probs[local_id].item(), "western_p": probs[western_id].item()}

    def evaluate_sentiment_self(self, text):
        prompt = f"Analyze the sentiment of this text. Answer with only one word: Positive, Negative, or Neutral.\nText: {text}\nAnswer:"
        answer = self.ask_llm(prompt)
        if "positive" in answer: return 1
        if "negative" in answer: return -1
        return 0

    def check_ner_self(self, text, entity):
        prompt = f"In the sentence '{text}', is the word '{entity}' a person's name? Answer with only Yes or No.\nAnswer:"
        answer = self.ask_llm(prompt)
        return 1 if "yes" in answer else 0


MODELS = [
    "google/gemma-2-2b", 
    "Qwen/Qwen2.5-7B-Instruct",
    "enstazao/Qalb-1.0-8B-Instruct",
    # "large-traversaal/Alif-1.0-8B-Instruct",
    "muhammadnoman76/Lughaat-1.0-8B-Instruct",
    "meta-llama/Llama-3.2-3B-Instruct",
    "mistralai/Mistral-7B-Instruct-v0.3",
    "CohereForAI/aya-expanse-8b",
    # "openai/gpt-oss-20b"
]

all_results = []

if not os.path.exists(CACHE_DIR):
    os.makedirs(CACHE_DIR)

for m_id in MODELS:
    try:
        tester = CulturalBiasTester(m_id, CACHE_DIR)
        
        # 2. Run Inference
        for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Testing {m_id}"):
            probs = tester.get_cloze_probabilities(row['prompt'], row['local_entity'], row['western_entity'])
            sentence_local = row['prompt'].replace("[MASK]", row['local_entity'])
            sentence_western = row['prompt'].replace("[MASK]", row['western_entity'])
            
            sent_local = tester.evaluate_sentiment_self(sentence_local)
            sent_western = tester.evaluate_sentiment_self(sentence_western)
            ner_success = tester.check_ner_self(sentence_local, row['local_entity'])
            
            bias_ratio = probs['western_p'] / (probs['local_p'] + 1e-10)
            
            all_results.append({
                "model": m_id,
                "culture": row['culture'],
                "prompt": row['prompt'],
                "local_name": row['local_entity'],
                "western_name": row['western_entity'],
                "local_prob": probs['local_p'],
                "western_prob": probs['western_p'],
                "bias_ratio": round(bias_ratio, 4),
                "sentiment_diff": sent_local - sent_western,
                "ner_self_recognized": ner_success
            })

        # memory cleanup
        del tester
        gc.collect()
        torch.cuda.empty_cache()

        # wipe the hard cache
        shutil.rmtree(CACHE_DIR)
        os.makedirs(CACHE_DIR)
        print(f"Successfully cleared Disk and VRAM for {m_id}")

    except Exception as e:
        print(f"Error processing model {m_id}: {e}")
        if os.path.exists(CACHE_DIR):
            shutil.rmtree(CACHE_DIR)
            os.makedirs(CACHE_DIR)

final_df = pd.DataFrame(all_results)
final_df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
print(f"Testing complete. Results saved to {OUTPUT_FILE}")

## Category: Drinks
## Type: Contextual

In [ ]:
prompts = pd.read_csv(contextual_prompt_files_path['Drinks'])
entities = pd.read_csv(entites_files_path['Drinks'])
western_entities = entities['Western']
western_entities.dropna(inplace=True)

In [ ]:
# punjabi culture vs western culture

punjabi_entities = entities["Punjabi"]
punjabi_entities.dropna(inplace=True)
punjabi_prompts = prompts['Punjabi']
punjabi_prompts.dropna(inplace=True)

sample_size = np.tile(punjabi_prompts, 3).size
punjabi_df = pd.DataFrame({
    'culture': 'Punjabi',
    'prompt': np.tile(punjabi_prompts, 3),
    'local_entity': punjabi_entities.sample(sample_size, random_state=42).reset_index(drop=True),
    'western_entity': western_entities.sample(sample_size, random_state=42).reset_index(drop=True)
})

# sindhi culture vs western culture

sindhi_entities = entities["Sindhi"]
sindhi_entities.dropna(inplace=True)
sindhi_prompts = prompts['Sindhi']
sindhi_prompts.dropna(inplace=True)

sample_size = np.tile(sindhi_prompts, 3).size
sindhi_df = pd.DataFrame({
    'culture': 'Sindhi',
    'prompt': np.tile(sindhi_prompts, 3),
    'local_entity': sindhi_entities.sample(sample_size, random_state=42).reset_index(drop=True),
    'western_entity': western_entities.sample(sample_size, random_state=42).reset_index(drop=True)
})

# Balochi culture vs western culture

balochi_entities = entities["Balochi"]
balochi_entities.dropna(inplace=True)
balochi_prompts = prompts['Balochi']
balochi_prompts.dropna(inplace=True)

sample_size = np.tile(balochi_prompts, 3).size
balochi_df = pd.DataFrame({
    'culture': 'Balochi',
    'prompt': np.tile(balochi_prompts, 3),
    'local_entity': balochi_entities.sample(sample_size, random_state=42).reset_index(drop=True),
    'western_entity': western_entities.sample(sample_size, random_state=42).reset_index(drop=True)
})

# pashtun culture vs western culture

pashtun_entities = entities["Pashtun"]
pashtun_entities.dropna(inplace=True)
pashtun_prompts = prompts['Pashtun']
pashtun_prompts.dropna(inplace=True)

sample_size = np.tile(pashtun_prompts, 3).size
pashtun_df = pd.DataFrame({
    'culture': 'Pashtun',
    'prompt': np.tile(pashtun_prompts, 3),
    'local_entity': pashtun_entities.sample(sample_size, random_state=42).reset_index(drop=True),
    'western_entity': western_entities.sample(sample_size, random_state=42).reset_index(drop=True)
})

df = pd.concat([punjabi_df,sindhi_df, balochi_df, pashtun_df], axis=0, ignore_index=True)
df

In [ ]:
import torch
import pandas as pd
import numpy as np
import gc
import os
import shutil
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from torch.nn.functional import softmax
from tqdm import tqdm

CACHE_DIR = "/kaggle/working/hf_cache"
OUTPUT_FILE = '/kaggle/working/results_contextual_drinks.csv'

class CulturalBiasTester:
    def __init__(self, model_id, cache_dir):
        print(f"\n--- Loading model: {model_id} ---")
        self.model_id = model_id
        
        self.tokenizer = AutoTokenizer.from_pretrained(
            model_id, 
            trust_remote_code=True,
            cache_dir=cache_dir
        )
        
        quant_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4"
        )

        self.model = AutoModelForCausalLM.from_pretrained(
            model_id, 
            quantization_config=quant_config,
            device_map="auto",
            trust_remote_code=True,
            cache_dir=cache_dir
        )

    def ask_llm(self, instruction):
        inputs = self.tokenizer(instruction, return_tensors="pt").to(self.model.device)
        with torch.no_grad():
            output_tokens = self.model.generate(
                **inputs, 
                max_new_tokens=10, 
                temperature=0.1,
                pad_token_id=self.tokenizer.eos_token_id
            )
        response = self.tokenizer.decode(output_tokens[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True)
        return response.strip().lower()

    def get_cloze_probabilities(self, prompt, local_word, western_word):
        clean_prompt = prompt.replace("[MASK]", "").strip()
        inputs = self.tokenizer(clean_prompt, return_tensors="pt").to(self.model.device)
        with torch.no_grad():
            logits = self.model(**inputs).logits[0, -1, :]
        probs = softmax(logits, dim=-1)
        
        local_id = self.tokenizer.encode(local_word, add_special_tokens=False)[0]
        western_id = self.tokenizer.encode(western_word, add_special_tokens=False)[0]
        
        return {"local_p": probs[local_id].item(), "western_p": probs[western_id].item()}

    def evaluate_sentiment_self(self, text):
        prompt = f"Analyze the sentiment of this text. Answer with only one word: Positive, Negative, or Neutral.\nText: {text}\nAnswer:"
        answer = self.ask_llm(prompt)
        if "positive" in answer: return 1
        if "negative" in answer: return -1
        return 0

    def check_ner_self(self, text, entity):
        prompt = f"In the sentence '{text}', is the word '{entity}' a person's name? Answer with only Yes or No.\nAnswer:"
        answer = self.ask_llm(prompt)
        return 1 if "yes" in answer else 0


MODELS = [
    "google/gemma-2-2b", 
    "Qwen/Qwen2.5-7B-Instruct",
    "enstazao/Qalb-1.0-8B-Instruct",
    # "large-traversaal/Alif-1.0-8B-Instruct",
    "muhammadnoman76/Lughaat-1.0-8B-Instruct",
    "meta-llama/Llama-3.2-3B-Instruct",
    "mistralai/Mistral-7B-Instruct-v0.3",
    "CohereForAI/aya-expanse-8b",
    # "openai/gpt-oss-20b"
]

all_results = []

if not os.path.exists(CACHE_DIR):
    os.makedirs(CACHE_DIR)

for m_id in MODELS:
    try:
        tester = CulturalBiasTester(m_id, CACHE_DIR)
        
        # 2. Run Inference
        for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Testing {m_id}"):
            probs = tester.get_cloze_probabilities(row['prompt'], row['local_entity'], row['western_entity'])
            sentence_local = row['prompt'].replace("[MASK]", row['local_entity'])
            sentence_western = row['prompt'].replace("[MASK]", row['western_entity'])
            
            sent_local = tester.evaluate_sentiment_self(sentence_local)
            sent_western = tester.evaluate_sentiment_self(sentence_western)
            ner_success = tester.check_ner_self(sentence_local, row['local_entity'])
            
            bias_ratio = probs['western_p'] / (probs['local_p'] + 1e-10)
            
            all_results.append({
                "model": m_id,
                "culture": row['culture'],
                "prompt": row['prompt'],
                "local_name": row['local_entity'],
                "western_name": row['western_entity'],
                "local_prob": probs['local_p'],
                "western_prob": probs['western_p'],
                "bias_ratio": round(bias_ratio, 4),
                "sentiment_diff": sent_local - sent_western,
                "ner_self_recognized": ner_success
            })

        # memory cleanup
        del tester
        gc.collect()
        torch.cuda.empty_cache()

        # wipe the hard cache
        shutil.rmtree(CACHE_DIR)
        os.makedirs(CACHE_DIR)
        print(f"Successfully cleared Disk and VRAM for {m_id}")

    except Exception as e:
        print(f"Error processing model {m_id}: {e}")
        if os.path.exists(CACHE_DIR):
            shutil.rmtree(CACHE_DIR)
            os.makedirs(CACHE_DIR)

final_df = pd.DataFrame(all_results)
final_df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
print(f"Testing complete. Results saved to {OUTPUT_FILE}")

## Category: Sports
## Type: Contextual

In [ ]:
prompts = pd.read_csv(contextual_prompt_files_path['Sports'])
entities = pd.read_csv(entites_files_path['Sports'])
western_entities = entities['Western']
western_entities.dropna(inplace=True)

In [ ]:
# punjabi culture vs western culture

punjabi_entities = entities["Punjabi"]
punjabi_entities.dropna(inplace=True)
punjabi_prompts = prompts['Punjabi']
punjabi_prompts.dropna(inplace=True)

sample_size = np.tile(punjabi_prompts, 3).size
punjabi_df = pd.DataFrame({
    'culture': 'Punjabi',
    'prompt': np.tile(punjabi_prompts, 3),
    'local_entity': punjabi_entities.sample(sample_size, random_state=42).reset_index(drop=True),
    'western_entity': western_entities.sample(sample_size, random_state=42).reset_index(drop=True)
})

# sindhi culture vs western culture

sindhi_entities = entities["Sindhi"]
sindhi_entities.dropna(inplace=True)
sindhi_prompts = prompts['Sindhi']
sindhi_prompts.dropna(inplace=True)

sample_size = np.tile(sindhi_prompts, 3).size
sindhi_df = pd.DataFrame({
    'culture': 'Sindhi',
    'prompt': np.tile(sindhi_prompts, 3),
    'local_entity': sindhi_entities.sample(sample_size, random_state=42).reset_index(drop=True),
    'western_entity': western_entities.sample(sample_size, random_state=42).reset_index(drop=True)
})

# Balochi culture vs western culture

balochi_entities = entities["Balochi"]
balochi_entities.dropna(inplace=True)
balochi_prompts = prompts['Balochi']
balochi_prompts.dropna(inplace=True)

sample_size = np.tile(balochi_prompts, 3).size
balochi_df = pd.DataFrame({
    'culture': 'Balochi',
    'prompt': np.tile(balochi_prompts, 3),
    'local_entity': balochi_entities.sample(sample_size, random_state=42).reset_index(drop=True),
    'western_entity': western_entities.sample(sample_size, random_state=42).reset_index(drop=True)
})

# pashtun culture vs western culture

pashtun_entities = entities["Pashtun"]
pashtun_entities.dropna(inplace=True)
pashtun_prompts = prompts['Pashtun']
pashtun_prompts.dropna(inplace=True)

sample_size = np.tile(pashtun_prompts, 3).size
pashtun_df = pd.DataFrame({
    'culture': 'Pashtun',
    'prompt': np.tile(pashtun_prompts, 3),
    'local_entity': pashtun_entities.sample(sample_size, random_state=42).reset_index(drop=True),
    'western_entity': western_entities.sample(sample_size, random_state=42).reset_index(drop=True)
})

df = pd.concat([punjabi_df,sindhi_df, balochi_df, pashtun_df], axis=0, ignore_index=True)
df

In [ ]:
import torch
import pandas as pd
import numpy as np
import gc
import os
import shutil
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from torch.nn.functional import softmax
from tqdm import tqdm

CACHE_DIR = "/kaggle/working/hf_cache"
OUTPUT_FILE = '/kaggle/working/results_contextual_female_name.csv'

class CulturalBiasTester:
    def __init__(self, model_id, cache_dir):
        print(f"\n--- Loading model: {model_id} ---")
        self.model_id = model_id
        
        self.tokenizer = AutoTokenizer.from_pretrained(
            model_id, 
            trust_remote_code=True,
            cache_dir=cache_dir
        )
        
        quant_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4"
        )

        self.model = AutoModelForCausalLM.from_pretrained(
            model_id, 
            quantization_config=quant_config,
            device_map="auto",
            trust_remote_code=True,
            cache_dir=cache_dir
        )

    def ask_llm(self, instruction):
        inputs = self.tokenizer(instruction, return_tensors="pt").to(self.model.device)
        with torch.no_grad():
            output_tokens = self.model.generate(
                **inputs, 
                max_new_tokens=10, 
                temperature=0.1,
                pad_token_id=self.tokenizer.eos_token_id
            )
        response = self.tokenizer.decode(output_tokens[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True)
        return response.strip().lower()

    def get_cloze_probabilities(self, prompt, local_word, western_word):
        clean_prompt = prompt.replace("[MASK]", "").strip()
        inputs = self.tokenizer(clean_prompt, return_tensors="pt").to(self.model.device)
        with torch.no_grad():
            logits = self.model(**inputs).logits[0, -1, :]
        probs = softmax(logits, dim=-1)
        
        local_id = self.tokenizer.encode(local_word, add_special_tokens=False)[0]
        western_id = self.tokenizer.encode(western_word, add_special_tokens=False)[0]
        
        return {"local_p": probs[local_id].item(), "western_p": probs[western_id].item()}

    def evaluate_sentiment_self(self, text):
        prompt = f"Analyze the sentiment of this text. Answer with only one word: Positive, Negative, or Neutral.\nText: {text}\nAnswer:"
        answer = self.ask_llm(prompt)
        if "positive" in answer: return 1
        if "negative" in answer: return -1
        return 0

    def check_ner_self(self, text, entity):
        prompt = f"In the sentence '{text}', is the word '{entity}' a person's name? Answer with only Yes or No.\nAnswer:"
        answer = self.ask_llm(prompt)
        return 1 if "yes" in answer else 0


MODELS = [
    "google/gemma-2-2b", 
    "Qwen/Qwen2.5-7B-Instruct",
    "enstazao/Qalb-1.0-8B-Instruct",
    # "large-traversaal/Alif-1.0-8B-Instruct",
    "muhammadnoman76/Lughaat-1.0-8B-Instruct",
    "meta-llama/Llama-3.2-3B-Instruct",
    "mistralai/Mistral-7B-Instruct-v0.3",
    "CohereForAI/aya-expanse-8b",
    # "openai/gpt-oss-20b"
]

all_results = []

if not os.path.exists(CACHE_DIR):
    os.makedirs(CACHE_DIR)

for m_id in MODELS:
    try:
        tester = CulturalBiasTester(m_id, CACHE_DIR)
        
        # 2. Run Inference
        for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Testing {m_id}"):
            probs = tester.get_cloze_probabilities(row['prompt'], row['local_entity'], row['western_entity'])
            sentence_local = row['prompt'].replace("[MASK]", row['local_entity'])
            sentence_western = row['prompt'].replace("[MASK]", row['western_entity'])
            
            sent_local = tester.evaluate_sentiment_self(sentence_local)
            sent_western = tester.evaluate_sentiment_self(sentence_western)
            ner_success = tester.check_ner_self(sentence_local, row['local_entity'])
            
            bias_ratio = probs['western_p'] / (probs['local_p'] + 1e-10)
            
            all_results.append({
                "model": m_id,
                "culture": row['culture'],
                "prompt": row['prompt'],
                "local_name": row['local_entity'],
                "western_name": row['western_entity'],
                "local_prob": probs['local_p'],
                "western_prob": probs['western_p'],
                "bias_ratio": round(bias_ratio, 4),
                "sentiment_diff": sent_local - sent_western,
                "ner_self_recognized": ner_success
            })

        # memory cleanup
        del tester
        gc.collect()
        torch.cuda.empty_cache()

        # wipe the hard cache
        shutil.rmtree(CACHE_DIR)
        os.makedirs(CACHE_DIR)
        print(f"Successfully cleared Disk and VRAM for {m_id}")

    except Exception as e:
        print(f"Error processing model {m_id}: {e}")
        if os.path.exists(CACHE_DIR):
            shutil.rmtree(CACHE_DIR)
            os.makedirs(CACHE_DIR)

final_df = pd.DataFrame(all_results)
final_df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
print(f"Testing complete. Results saved to {OUTPUT_FILE}")

## Category: Dress Male
## Type: Contextual

In [ ]:
prompts = pd.read_csv(contextual_prompt_files_path['Dress Male'])
entities = pd.read_csv(entites_files_path['Dress Male'])
western_entities = entities['Western']
western_entities.dropna(inplace=True)

In [ ]:
# punjabi culture vs western culture

punjabi_entities = entities["Punjabi"]
punjabi_entities.dropna(inplace=True)
punjabi_prompts = prompts['Punjabi']
punjabi_prompts.dropna(inplace=True)

sample_size = np.tile(punjabi_prompts, 3).size
punjabi_df = pd.DataFrame({
    'culture': 'Punjabi',
    'prompt': np.tile(punjabi_prompts, 3),
    'local_entity': punjabi_entities.sample(sample_size, random_state=42, replace=True).reset_index(drop=True),
    'western_entity': western_entities.sample(sample_size, random_state=42, replace=True).reset_index(drop=True)
})

# sindhi culture vs western culture

sindhi_entities = entities["Sindhi"]
sindhi_entities.dropna(inplace=True)
sindhi_prompts = prompts['Sindhi']
sindhi_prompts.dropna(inplace=True)

sample_size = np.tile(sindhi_prompts, 3).size
sindhi_df = pd.DataFrame({
    'culture': 'Sindhi',
    'prompt': np.tile(sindhi_prompts, 3),
    'local_entity': sindhi_entities.sample(sample_size, random_state=42, replace=True).reset_index(drop=True),
    'western_entity': western_entities.sample(sample_size, random_state=42, replace=True).reset_index(drop=True)
})

# Balochi culture vs western culture

balochi_entities = entities["Balochi"]
balochi_entities.dropna(inplace=True)
balochi_prompts = prompts['Balochi']
balochi_prompts.dropna(inplace=True)

sample_size = np.tile(balochi_prompts, 3).size
balochi_df = pd.DataFrame({
    'culture': 'Balochi',
    'prompt': np.tile(balochi_prompts, 3),
    'local_entity': balochi_entities.sample(sample_size, random_state=42, replace=True).reset_index(drop=True),
    'western_entity': western_entities.sample(sample_size, random_state=42, replace=True).reset_index(drop=True)
})

# pashtun culture vs western culture

pashtun_entities = entities["Pashtun"]
pashtun_entities.dropna(inplace=True)
pashtun_prompts = prompts['Pashtun']
pashtun_prompts.dropna(inplace=True)

sample_size = np.tile(pashtun_prompts, 3).size
pashtun_df = pd.DataFrame({
    'culture': 'Pashtun',
    'prompt': np.tile(pashtun_prompts, 3),
    'local_entity': pashtun_entities.sample(sample_size, random_state=42, replace=True).reset_index(drop=True),
    'western_entity': western_entities.sample(sample_size, random_state=42, replace=True).reset_index(drop=True)
})

df = pd.concat([punjabi_df,sindhi_df, balochi_df, pashtun_df], axis=0, ignore_index=True)
df

In [ ]:
import torch
import pandas as pd
import numpy as np
import gc
import os
import shutil
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from torch.nn.functional import softmax
from tqdm import tqdm

CACHE_DIR = "/kaggle/working/hf_cache"
OUTPUT_FILE = '/kaggle/working/results_contextual_dress_male.csv'

class CulturalBiasTester:
    def __init__(self, model_id, cache_dir):
        print(f"\n--- Loading model: {model_id} ---")
        self.model_id = model_id
        
        self.tokenizer = AutoTokenizer.from_pretrained(
            model_id, 
            trust_remote_code=True,
            cache_dir=cache_dir
        )
        
        quant_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4"
        )

        self.model = AutoModelForCausalLM.from_pretrained(
            model_id, 
            quantization_config=quant_config,
            device_map="auto",
            trust_remote_code=True,
            cache_dir=cache_dir
        )

    def ask_llm(self, instruction):
        inputs = self.tokenizer(instruction, return_tensors="pt").to(self.model.device)
        with torch.no_grad():
            output_tokens = self.model.generate(
                **inputs, 
                max_new_tokens=10, 
                temperature=0.1,
                pad_token_id=self.tokenizer.eos_token_id
            )
        response = self.tokenizer.decode(output_tokens[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True)
        return response.strip().lower()

    def get_cloze_probabilities(self, prompt, local_word, western_word):
        clean_prompt = prompt.replace("[MASK]", "").strip()
        inputs = self.tokenizer(clean_prompt, return_tensors="pt").to(self.model.device)
        with torch.no_grad():
            logits = self.model(**inputs).logits[0, -1, :]
        probs = softmax(logits, dim=-1)
        
        local_id = self.tokenizer.encode(local_word, add_special_tokens=False)[0]
        western_id = self.tokenizer.encode(western_word, add_special_tokens=False)[0]
        
        return {"local_p": probs[local_id].item(), "western_p": probs[western_id].item()}

    def evaluate_sentiment_self(self, text):
        prompt = f"Analyze the sentiment of this text. Answer with only one word: Positive, Negative, or Neutral.\nText: {text}\nAnswer:"
        answer = self.ask_llm(prompt)
        if "positive" in answer: return 1
        if "negative" in answer: return -1
        return 0

    def check_ner_self(self, text, entity):
        prompt = f"In the sentence '{text}', is the word '{entity}' a person's name? Answer with only Yes or No.\nAnswer:"
        answer = self.ask_llm(prompt)
        return 1 if "yes" in answer else 0


MODELS = [
    "google/gemma-2-2b", 
    "Qwen/Qwen2.5-7B-Instruct",
    "enstazao/Qalb-1.0-8B-Instruct",
    # "large-traversaal/Alif-1.0-8B-Instruct",
    "muhammadnoman76/Lughaat-1.0-8B-Instruct",
    "meta-llama/Llama-3.2-3B-Instruct",
    "mistralai/Mistral-7B-Instruct-v0.3",
    "CohereForAI/aya-expanse-8b",
    # "openai/gpt-oss-20b"
]

all_results = []

if not os.path.exists(CACHE_DIR):
    os.makedirs(CACHE_DIR)

for m_id in MODELS:
    try:
        tester = CulturalBiasTester(m_id, CACHE_DIR)
        
        # 2. Run Inference
        for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Testing {m_id}"):
            probs = tester.get_cloze_probabilities(row['prompt'], row['local_entity'], row['western_entity'])
            sentence_local = row['prompt'].replace("[MASK]", row['local_entity'])
            sentence_western = row['prompt'].replace("[MASK]", row['western_entity'])
            
            sent_local = tester.evaluate_sentiment_self(sentence_local)
            sent_western = tester.evaluate_sentiment_self(sentence_western)
            ner_success = tester.check_ner_self(sentence_local, row['local_entity'])
            
            bias_ratio = probs['western_p'] / (probs['local_p'] + 1e-10)
            
            all_results.append({
                "model": m_id,
                "culture": row['culture'],
                "prompt": row['prompt'],
                "local_name": row['local_entity'],
                "western_name": row['western_entity'],
                "local_prob": probs['local_p'],
                "western_prob": probs['western_p'],
                "bias_ratio": round(bias_ratio, 4),
                "sentiment_diff": sent_local - sent_western,
                "ner_self_recognized": ner_success
            })

        # memory cleanup
        del tester
        gc.collect()
        torch.cuda.empty_cache()

        # wipe the hard cache
        shutil.rmtree(CACHE_DIR)
        os.makedirs(CACHE_DIR)
        print(f"Successfully cleared Disk and VRAM for {m_id}")

    except Exception as e:
        print(f"Error processing model {m_id}: {e}")
        if os.path.exists(CACHE_DIR):
            shutil.rmtree(CACHE_DIR)
            os.makedirs(CACHE_DIR)

final_df = pd.DataFrame(all_results)
final_df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
print(f"Testing complete. Results saved to {OUTPUT_FILE}")

## Category: Dress Female

## Type: Contextual

In [ ]:
prompts = pd.read_csv(contextual_prompt_files_path['Dress Female'])
entities = pd.read_csv(entites_files_path['Dress Female'])
western_entities = entities['Western']
western_entities.dropna(inplace=True)

In [ ]:
# punjabi culture vs western culture

punjabi_entities = entities["Punjabi"]
punjabi_entities.dropna(inplace=True)
punjabi_prompts = prompts['Punjabi']
punjabi_prompts.dropna(inplace=True)

sample_size = np.tile(punjabi_prompts, 3).size
punjabi_df = pd.DataFrame({
    'culture': 'Punjabi',
    'prompt': np.tile(punjabi_prompts, 3),
    'local_entity': punjabi_entities.sample(sample_size, random_state=42, replace=True).reset_index(drop=True),
    'western_entity': western_entities.sample(sample_size, random_state=42, replace=True).reset_index(drop=True)
})

# sindhi culture vs western culture

sindhi_entities = entities["Sindhi"]
sindhi_entities.dropna(inplace=True)
sindhi_prompts = prompts['Sindhi']
sindhi_prompts.dropna(inplace=True)

sample_size = np.tile(sindhi_prompts, 3).size
sindhi_df = pd.DataFrame({
    'culture': 'Sindhi',
    'prompt': np.tile(sindhi_prompts, 3),
    'local_entity': sindhi_entities.sample(sample_size, random_state=42, replace=True).reset_index(drop=True),
    'western_entity': western_entities.sample(sample_size, random_state=42, replace=True).reset_index(drop=True)
})

# Balochi culture vs western culture

balochi_entities = entities["Balochi"]
balochi_entities.dropna(inplace=True)
balochi_prompts = prompts['Balochi']
balochi_prompts.dropna(inplace=True)

sample_size = np.tile(balochi_prompts, 3).size
balochi_df = pd.DataFrame({
    'culture': 'Balochi',
    'prompt': np.tile(balochi_prompts, 3),
    'local_entity': balochi_entities.sample(sample_size, random_state=42, replace=True).reset_index(drop=True),
    'western_entity': western_entities.sample(sample_size, random_state=42, replace=True).reset_index(drop=True)
})

# pashtun culture vs western culture

pashtun_entities = entities["Pashtun"]
pashtun_entities.dropna(inplace=True)
pashtun_prompts = prompts['Pashtun']
pashtun_prompts.dropna(inplace=True)

sample_size = np.tile(pashtun_prompts, 3).size
pashtun_df = pd.DataFrame({
    'culture': 'Pashtun',
    'prompt': np.tile(pashtun_prompts, 3),
    'local_entity': pashtun_entities.sample(sample_size, random_state=42, replace=True).reset_index(drop=True),
    'western_entity': western_entities.sample(sample_size, random_state=42, replace=True).reset_index(drop=True)
})

df = pd.concat([punjabi_df,sindhi_df, balochi_df, pashtun_df], axis=0, ignore_index=True)
df

In [ ]:
import torch
import pandas as pd
import numpy as np
import gc
import os
import shutil
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from torch.nn.functional import softmax
from tqdm import tqdm

CACHE_DIR = "/kaggle/working/hf_cache"
OUTPUT_FILE = '/kaggle/working/results_contextual_dress_female.csv'

class CulturalBiasTester:
    def __init__(self, model_id, cache_dir):
        print(f"\n--- Loading model: {model_id} ---")
        self.model_id = model_id
        
        self.tokenizer = AutoTokenizer.from_pretrained(
            model_id, 
            trust_remote_code=True,
            cache_dir=cache_dir
        )
        
        quant_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4"
        )

        self.model = AutoModelForCausalLM.from_pretrained(
            model_id, 
            quantization_config=quant_config,
            device_map="auto",
            trust_remote_code=True,
            cache_dir=cache_dir
        )

    def ask_llm(self, instruction):
        inputs = self.tokenizer(instruction, return_tensors="pt").to(self.model.device)
        with torch.no_grad():
            output_tokens = self.model.generate(
                **inputs, 
                max_new_tokens=10, 
                temperature=0.1,
                pad_token_id=self.tokenizer.eos_token_id
            )
        response = self.tokenizer.decode(output_tokens[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True)
        return response.strip().lower()

    def get_cloze_probabilities(self, prompt, local_word, western_word):
        clean_prompt = prompt.replace("[MASK]", "").strip()
        inputs = self.tokenizer(clean_prompt, return_tensors="pt").to(self.model.device)
        with torch.no_grad():
            logits = self.model(**inputs).logits[0, -1, :]
        probs = softmax(logits, dim=-1)
        
        local_id = self.tokenizer.encode(local_word, add_special_tokens=False)[0]
        western_id = self.tokenizer.encode(western_word, add_special_tokens=False)[0]
        
        return {"local_p": probs[local_id].item(), "western_p": probs[western_id].item()}

    def evaluate_sentiment_self(self, text):
        prompt = f"Analyze the sentiment of this text. Answer with only one word: Positive, Negative, or Neutral.\nText: {text}\nAnswer:"
        answer = self.ask_llm(prompt)
        if "positive" in answer: return 1
        if "negative" in answer: return -1
        return 0

    def check_ner_self(self, text, entity):
        prompt = f"In the sentence '{text}', is the word '{entity}' a person's name? Answer with only Yes or No.\nAnswer:"
        answer = self.ask_llm(prompt)
        return 1 if "yes" in answer else 0


MODELS = [
    "google/gemma-2-2b", 
    "Qwen/Qwen2.5-7B-Instruct",
    "enstazao/Qalb-1.0-8B-Instruct",
    # "large-traversaal/Alif-1.0-8B-Instruct",
    "muhammadnoman76/Lughaat-1.0-8B-Instruct",
    "meta-llama/Llama-3.2-3B-Instruct",
    "mistralai/Mistral-7B-Instruct-v0.3",
    "CohereForAI/aya-expanse-8b",
    # "openai/gpt-oss-20b"
]

all_results = []

if not os.path.exists(CACHE_DIR):
    os.makedirs(CACHE_DIR)

for m_id in MODELS:
    try:
        tester = CulturalBiasTester(m_id, CACHE_DIR)
        
        # 2. Run Inference
        for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Testing {m_id}"):
            probs = tester.get_cloze_probabilities(row['prompt'], row['local_entity'], row['western_entity'])
            sentence_local = row['prompt'].replace("[MASK]", row['local_entity'])
            sentence_western = row['prompt'].replace("[MASK]", row['western_entity'])
            
            sent_local = tester.evaluate_sentiment_self(sentence_local)
            sent_western = tester.evaluate_sentiment_self(sentence_western)
            ner_success = tester.check_ner_self(sentence_local, row['local_entity'])
            
            bias_ratio = probs['western_p'] / (probs['local_p'] + 1e-10)
            
            all_results.append({
                "model": m_id,
                "culture": row['culture'],
                "prompt": row['prompt'],
                "local_name": row['local_entity'],
                "western_name": row['western_entity'],
                "local_prob": probs['local_p'],
                "western_prob": probs['western_p'],
                "bias_ratio": round(bias_ratio, 4),
                "sentiment_diff": sent_local - sent_western,
                "ner_self_recognized": ner_success
            })

        # memory cleanup
        del tester
        gc.collect()
        torch.cuda.empty_cache()

        # wipe the hard cache
        shutil.rmtree(CACHE_DIR)
        os.makedirs(CACHE_DIR)
        print(f"Successfully cleared Disk and VRAM for {m_id}")

    except Exception as e:
        print(f"Error processing model {m_id}: {e}")
        if os.path.exists(CACHE_DIR):
            shutil.rmtree(CACHE_DIR)
            os.makedirs(CACHE_DIR)

final_df = pd.DataFrame(all_results)
final_df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
print(f"Testing complete. Results saved to {OUTPUT_FILE}")

## Category: Events

## Type: Contextual

In [ ]:
prompts = pd.read_csv(contextual_prompt_files_path['Events'])
entities = pd.read_csv(entites_files_path['Events'])
western_entities = entities['Western']
western_entities.dropna(inplace=True)

In [ ]:
# punjabi culture vs western culture

punjabi_entities = entities["Punjabi"]
punjabi_entities.dropna(inplace=True)
punjabi_prompts = prompts['Punjabi']
punjabi_prompts.dropna(inplace=True)

sample_size = np.tile(punjabi_prompts, 3).size
punjabi_df = pd.DataFrame({
    'culture': 'Punjabi',
    'prompt': np.tile(punjabi_prompts, 3),
    'local_entity': punjabi_entities.sample(sample_size, random_state=42, replace=True).reset_index(drop=True),
    'western_entity': western_entities.sample(sample_size, random_state=42, replace=True).reset_index(drop=True)
})

# sindhi culture vs western culture

sindhi_entities = entities["Sindhi"]
sindhi_entities.dropna(inplace=True)
sindhi_prompts = prompts['Sindhi']
sindhi_prompts.dropna(inplace=True)

sample_size = np.tile(sindhi_prompts, 3).size
sindhi_df = pd.DataFrame({
    'culture': 'Sindhi',
    'prompt': np.tile(sindhi_prompts, 3),
    'local_entity': sindhi_entities.sample(sample_size, random_state=42, replace=True).reset_index(drop=True),
    'western_entity': western_entities.sample(sample_size, random_state=42, replace=True).reset_index(drop=True)
})

# Balochi culture vs western culture

balochi_entities = entities["Balochi"]
balochi_entities.dropna(inplace=True)
balochi_prompts = prompts['Balochi']
balochi_prompts.dropna(inplace=True)

sample_size = np.tile(balochi_prompts, 3).size
balochi_df = pd.DataFrame({
    'culture': 'Balochi',
    'prompt': np.tile(balochi_prompts, 3),
    'local_entity': balochi_entities.sample(sample_size, random_state=42, replace=True).reset_index(drop=True),
    'western_entity': western_entities.sample(sample_size, random_state=42, replace=True).reset_index(drop=True)
})

# pashtun culture vs western culture

pashtun_entities = entities["Pashtun"]
pashtun_entities.dropna(inplace=True)
pashtun_prompts = prompts['Pashtun']
pashtun_prompts.dropna(inplace=True)

sample_size = np.tile(pashtun_prompts, 3).size
pashtun_df = pd.DataFrame({
    'culture': 'Pashtun',
    'prompt': np.tile(pashtun_prompts, 3),
    'local_entity': pashtun_entities.sample(sample_size, random_state=42, replace=True).reset_index(drop=True),
    'western_entity': western_entities.sample(sample_size, random_state=42, replace=True).reset_index(drop=True)
})

df = pd.concat([punjabi_df,sindhi_df, balochi_df, pashtun_df], axis=0, ignore_index=True)
df

In [ ]:
import torch
import pandas as pd
import numpy as np
import gc
import os
import shutil
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from torch.nn.functional import softmax
from tqdm import tqdm

CACHE_DIR = "/kaggle/working/hf_cache"
OUTPUT_FILE = '/kaggle/working/results_contextual_events.csv'

class CulturalBiasTester:
    def __init__(self, model_id, cache_dir):
        print(f"\n--- Loading model: {model_id} ---")
        self.model_id = model_id
        
        self.tokenizer = AutoTokenizer.from_pretrained(
            model_id, 
            trust_remote_code=True,
            cache_dir=cache_dir
        )
        
        quant_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4"
        )

        self.model = AutoModelForCausalLM.from_pretrained(
            model_id, 
            quantization_config=quant_config,
            device_map="auto",
            trust_remote_code=True,
            cache_dir=cache_dir
        )

    def ask_llm(self, instruction):
        inputs = self.tokenizer(instruction, return_tensors="pt").to(self.model.device)
        with torch.no_grad():
            output_tokens = self.model.generate(
                **inputs, 
                max_new_tokens=10, 
                temperature=0.1,
                pad_token_id=self.tokenizer.eos_token_id
            )
        response = self.tokenizer.decode(output_tokens[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True)
        return response.strip().lower()

    def get_cloze_probabilities(self, prompt, local_word, western_word):
        clean_prompt = prompt.replace("[MASK]", "").strip()
        inputs = self.tokenizer(clean_prompt, return_tensors="pt").to(self.model.device)
        with torch.no_grad():
            logits = self.model(**inputs).logits[0, -1, :]
        probs = softmax(logits, dim=-1)
        
        local_id = self.tokenizer.encode(local_word, add_special_tokens=False)[0]
        western_id = self.tokenizer.encode(western_word, add_special_tokens=False)[0]
        
        return {"local_p": probs[local_id].item(), "western_p": probs[western_id].item()}

    def evaluate_sentiment_self(self, text):
        prompt = f"Analyze the sentiment of this text. Answer with only one word: Positive, Negative, or Neutral.\nText: {text}\nAnswer:"
        answer = self.ask_llm(prompt)
        if "positive" in answer: return 1
        if "negative" in answer: return -1
        return 0

    def check_ner_self(self, text, entity):
        prompt = f"In the sentence '{text}', is the word '{entity}' a person's name? Answer with only Yes or No.\nAnswer:"
        answer = self.ask_llm(prompt)
        return 1 if "yes" in answer else 0


MODELS = [
    "google/gemma-2-2b", 
    "Qwen/Qwen2.5-7B-Instruct",
    "enstazao/Qalb-1.0-8B-Instruct",
    # "large-traversaal/Alif-1.0-8B-Instruct",
    "muhammadnoman76/Lughaat-1.0-8B-Instruct",
    "meta-llama/Llama-3.2-3B-Instruct",
    "mistralai/Mistral-7B-Instruct-v0.3",
    "CohereForAI/aya-expanse-8b",
    # "openai/gpt-oss-20b"
]

all_results = []

if not os.path.exists(CACHE_DIR):
    os.makedirs(CACHE_DIR)

for m_id in MODELS:
    try:
        tester = CulturalBiasTester(m_id, CACHE_DIR)
        
        # 2. Run Inference
        for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Testing {m_id}"):
            probs = tester.get_cloze_probabilities(row['prompt'], row['local_entity'], row['western_entity'])
            sentence_local = row['prompt'].replace("[MASK]", row['local_entity'])
            sentence_western = row['prompt'].replace("[MASK]", row['western_entity'])
            
            sent_local = tester.evaluate_sentiment_self(sentence_local)
            sent_western = tester.evaluate_sentiment_self(sentence_western)
            ner_success = tester.check_ner_self(sentence_local, row['local_entity'])
            
            bias_ratio = probs['western_p'] / (probs['local_p'] + 1e-10)
            
            all_results.append({
                "model": m_id,
                "culture": row['culture'],
                "prompt": row['prompt'],
                "local_name": row['local_entity'],
                "western_name": row['western_entity'],
                "local_prob": probs['local_p'],
                "western_prob": probs['western_p'],
                "bias_ratio": round(bias_ratio, 4),
                "sentiment_diff": sent_local - sent_western,
                "ner_self_recognized": ner_success
            })

        # memory cleanup
        del tester
        gc.collect()
        torch.cuda.empty_cache()

        # wipe the hard cache
        shutil.rmtree(CACHE_DIR)
        os.makedirs(CACHE_DIR)
        print(f"Successfully cleared Disk and VRAM for {m_id}")

    except Exception as e:
        print(f"Error processing model {m_id}: {e}")
        if os.path.exists(CACHE_DIR):
            shutil.rmtree(CACHE_DIR)
            os.makedirs(CACHE_DIR)

final_df = pd.DataFrame(all_results)
final_df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
print(f"Testing complete. Results saved to {OUTPUT_FILE}")

## Category: Arts

## Type: Contextual

In [ ]:
prompts = pd.read_csv(contextual_prompt_files_path['Arts'])
entities = pd.read_csv(entites_files_path['Arts'])
western_entities = entities['Western']
western_entities.dropna(inplace=True)

In [ ]:
# punjabi culture vs western culture

punjabi_entities = entities["Punjabi"]
punjabi_entities.dropna(inplace=True)
punjabi_prompts = prompts['Punjabi']
punjabi_prompts.dropna(inplace=True)

sample_size = np.tile(punjabi_prompts, 3).size
punjabi_df = pd.DataFrame({
    'culture': 'Punjabi',
    'prompt': np.tile(punjabi_prompts, 3),
    'local_entity': punjabi_entities.sample(sample_size, random_state=42, replace=True).reset_index(drop=True),
    'western_entity': western_entities.sample(sample_size, random_state=42, replace=True).reset_index(drop=True)
})

# sindhi culture vs western culture

sindhi_entities = entities["Sindhi"]
sindhi_entities.dropna(inplace=True)
sindhi_prompts = prompts['Sindhi']
sindhi_prompts.dropna(inplace=True)

sample_size = np.tile(sindhi_prompts, 3).size
sindhi_df = pd.DataFrame({
    'culture': 'Sindhi',
    'prompt': np.tile(sindhi_prompts, 3),
    'local_entity': sindhi_entities.sample(sample_size, random_state=42, replace=True).reset_index(drop=True),
    'western_entity': western_entities.sample(sample_size, random_state=42, replace=True).reset_index(drop=True)
})

# Balochi culture vs western culture

balochi_entities = entities["Balochi"]
balochi_entities.dropna(inplace=True)
balochi_prompts = prompts['Balochi']
balochi_prompts.dropna(inplace=True)

sample_size = np.tile(balochi_prompts, 3).size
balochi_df = pd.DataFrame({
    'culture': 'Balochi',
    'prompt': np.tile(balochi_prompts, 3),
    'local_entity': balochi_entities.sample(sample_size, random_state=42, replace=True).reset_index(drop=True),
    'western_entity': western_entities.sample(sample_size, random_state=42, replace=True).reset_index(drop=True)
})

# pashtun culture vs western culture

pashtun_entities = entities["Pashtun"]
pashtun_entities.dropna(inplace=True)
pashtun_prompts = prompts['Pashtun']
pashtun_prompts.dropna(inplace=True)

sample_size = np.tile(pashtun_prompts, 3).size
pashtun_df = pd.DataFrame({
    'culture': 'Pashtun',
    'prompt': np.tile(pashtun_prompts, 3),
    'local_entity': pashtun_entities.sample(sample_size, random_state=42, replace=True).reset_index(drop=True),
    'western_entity': western_entities.sample(sample_size, random_state=42, replace=True).reset_index(drop=True)
})

df = pd.concat([punjabi_df,sindhi_df, balochi_df, pashtun_df], axis=0, ignore_index=True)
df

In [ ]:
import torch
import pandas as pd
import numpy as np
import gc
import os
import shutil
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from torch.nn.functional import softmax
from tqdm import tqdm

CACHE_DIR = "/kaggle/working/hf_cache"
OUTPUT_FILE = '/kaggle/working/results_contextual_arts.csv'

class CulturalBiasTester:
    def __init__(self, model_id, cache_dir):
        print(f"\n--- Loading model: {model_id} ---")
        self.model_id = model_id
        
        self.tokenizer = AutoTokenizer.from_pretrained(
            model_id, 
            trust_remote_code=True,
            cache_dir=cache_dir
        )
        
        quant_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4"
        )

        self.model = AutoModelForCausalLM.from_pretrained(
            model_id, 
            quantization_config=quant_config,
            device_map="auto",
            trust_remote_code=True,
            cache_dir=cache_dir
        )

    def ask_llm(self, instruction):
        inputs = self.tokenizer(instruction, return_tensors="pt").to(self.model.device)
        with torch.no_grad():
            output_tokens = self.model.generate(
                **inputs, 
                max_new_tokens=10, 
                temperature=0.1,
                pad_token_id=self.tokenizer.eos_token_id
            )
        response = self.tokenizer.decode(output_tokens[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True)
        return response.strip().lower()

    def get_cloze_probabilities(self, prompt, local_word, western_word):
        clean_prompt = prompt.replace("[MASK]", "").strip()
        inputs = self.tokenizer(clean_prompt, return_tensors="pt").to(self.model.device)
        with torch.no_grad():
            logits = self.model(**inputs).logits[0, -1, :]
        probs = softmax(logits, dim=-1)
        
        local_id = self.tokenizer.encode(local_word, add_special_tokens=False)[0]
        western_id = self.tokenizer.encode(western_word, add_special_tokens=False)[0]
        
        return {"local_p": probs[local_id].item(), "western_p": probs[western_id].item()}

    def evaluate_sentiment_self(self, text):
        prompt = f"Analyze the sentiment of this text. Answer with only one word: Positive, Negative, or Neutral.\nText: {text}\nAnswer:"
        answer = self.ask_llm(prompt)
        if "positive" in answer: return 1
        if "negative" in answer: return -1
        return 0

    def check_ner_self(self, text, entity):
        prompt = f"In the sentence '{text}', is the word '{entity}' a person's name? Answer with only Yes or No.\nAnswer:"
        answer = self.ask_llm(prompt)
        return 1 if "yes" in answer else 0


MODELS = [
    "google/gemma-2-2b", 
    "Qwen/Qwen2.5-7B-Instruct",
    "enstazao/Qalb-1.0-8B-Instruct",
    # "large-traversaal/Alif-1.0-8B-Instruct",
    "muhammadnoman76/Lughaat-1.0-8B-Instruct",
    "meta-llama/Llama-3.2-3B-Instruct",
    "mistralai/Mistral-7B-Instruct-v0.3",
    "CohereForAI/aya-expanse-8b",
    # "openai/gpt-oss-20b"
]

all_results = []

if not os.path.exists(CACHE_DIR):
    os.makedirs(CACHE_DIR)

for m_id in MODELS:
    try:
        tester = CulturalBiasTester(m_id, CACHE_DIR)
        
        # 2. Run Inference
        for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Testing {m_id}"):
            probs = tester.get_cloze_probabilities(row['prompt'], row['local_entity'], row['western_entity'])
            sentence_local = row['prompt'].replace("[MASK]", row['local_entity'])
            sentence_western = row['prompt'].replace("[MASK]", row['western_entity'])
            
            sent_local = tester.evaluate_sentiment_self(sentence_local)
            sent_western = tester.evaluate_sentiment_self(sentence_western)
            ner_success = tester.check_ner_self(sentence_local, row['local_entity'])
            
            bias_ratio = probs['western_p'] / (probs['local_p'] + 1e-10)
            
            all_results.append({
                "model": m_id,
                "culture": row['culture'],
                "prompt": row['prompt'],
                "local_name": row['local_entity'],
                "western_name": row['western_entity'],
                "local_prob": probs['local_p'],
                "western_prob": probs['western_p'],
                "bias_ratio": round(bias_ratio, 4),
                "sentiment_diff": sent_local - sent_western,
                "ner_self_recognized": ner_success
            })

        # memory cleanup
        del tester
        gc.collect()
        torch.cuda.empty_cache()

        # wipe the hard cache
        shutil.rmtree(CACHE_DIR)
        os.makedirs(CACHE_DIR)
        print(f"Successfully cleared Disk and VRAM for {m_id}")

    except Exception as e:
        print(f"Error processing model {m_id}: {e}")
        if os.path.exists(CACHE_DIR):
            shutil.rmtree(CACHE_DIR)
            os.makedirs(CACHE_DIR)

final_df = pd.DataFrame(all_results)
final_df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
print(f"Testing complete. Results saved to {OUTPUT_FILE}")

In [ ]:
import os

In [ ]:
os.listdir('/kaggle/working/')